# 2026 Spring "EE50045, QU50001: Introductory Quantum Mechanics"

###### Update history
###### 2025. 03. 27 : Minsu Jeong,  KAIST Electrical Engineering, Initial implementation of TMM code.
###### 2025. 04. 08 : Minsu Jeong,  KAIST Electrical Engineering, Updated potential shape (parabolic, etc).
###### 2025. 09. 01 : Minsu Jeong,  KAIST Electrical Engineering, Clean up and Update descriptions.

###### ref
###### - Levi, Applied Quantum Mechanics, 2ed (Cambridge, 2006) ch4
###### - Kittel, Introduction to Solid State Physics, 8ed (John Wiley & Sons, Inc) p536
###### - Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p44



# Transmission through finite barriers using Propagation matrix method (or, Transfer matrix mathod)

---


## Basic Concepts

-   **Segmentation of the System:**\
    The one-dimensional system is divided into regions where the
    potential $V(x)$ is constant. In each region, the time-independent
    Schrödinger equation has the general solution:\
    $$
    \psi(x) = A e^{ikx} + B e^{-ikx},
    $$\
    where the wave number is given by:\
    $$
    k = \sqrt{\frac{2m(E - V)}{\hbar^2}}.
    $$\
    For regions with $E < V$, the wave number $k$ becomes imaginary,
    representing evanescent (decaying) waves.

-   **Boundary Conditions:**\
    At the interface between two regions, both the wavefunction
    $\psi(x)$ and its derivative $\psi'(x)$ must remain continuous.
    These continuity conditions connect the coefficients $A$ and $B$ on
    either side of the interface.

-   **Transfer Matrix:**\
    For each region (or interface), a 2×2 transfer matrix $T_i$ is
    defined such that:\
    $$
    \begin{pmatrix} A_{i+1} \\ B_{i+1} \end{pmatrix} 
    = T_i \begin{pmatrix} A_i \\ B_i \end{pmatrix}.
    $$\
    The total transfer matrix $M$ of the entire structure is obtained as
    the product of individual matrices:\
    $$
    M = T_{N-1} T_{N-2} \cdots T_0.
    $$

-   **Transmission Coefficient:**\
    Assuming an incoming wave from the left-hand side ($A_0 = 1$) and no
    incoming wave from the right-hand side ($B_N = 0$), the transmitted
    and reflected amplitudes satisfy:\
    $$
    \begin{pmatrix} t \\ 0 \end{pmatrix}
    = M \begin{pmatrix} 1 \\ r \end{pmatrix}.
    $$\
    From this relation, the transmission amplitude is expressed as:\
    $$
    t = \frac{1}{M_{11}}.
    $$\
    The transmission coefficient is then given by:\
    $$
    T = |t|^2 \cdot \frac{k_N}{k_0},
    $$\
    where $k_0$ and $k_N$ are the wave numbers in the left and right
    regions, respectively. The ratio $\tfrac{k_N}{k_0}$ accounts for the
    difference in probability current density.\
    In the special case where the left and right potentials are equal
    ($k_N = k_0$), the formula reduces to:\
    $$
    T = |t|^2.
    $$

------------------------------------------------------------------------

## Procedure

1.  **Define Parameters:**

    -   Specify the particle energy $E$.\
    -   Define the potential $V_i$, width $d_i$ of each region, the
        effective mass $m$, and Planck's constant $\hbar$.

2.  **Calculate Wave Numbers:**\
    For each region $i$, compute:\
    $$
    k_i = \sqrt{\frac{2m(E - V_i)}{\hbar^2}},
    $$\
    with $k_i$ taken as imaginary if $E < V_i$.

3.  **Construct the Transfer Matrix for Each Region:**\
    For a region of width $d$ and wave number $k$, the transfer
    (propagation) matrix is typically:\
    $$
    M = \begin{pmatrix}
    \cos(k d) & \frac{i}{k}\sin(k d) \\
    i k \sin(k d) & \cos(k d)
    \end{pmatrix}.
    $$

4.  **Combine Transfer Matrices:**\
    Multiply the matrices in sequence to obtain the overall transfer
    matrix:\
    $$
    M_{\text{total}} = T_{N-1} T_{N-2} \cdots T_0.
    $$

5.  **Apply Boundary Conditions:**\
    Impose the conditions $A_0 = 1$ (unit amplitude incident wave) and
    $B_N = 0$ (no incoming wave from the right). Solve for the
    transmission amplitude $t$ using the total transfer matrix.

6.  **Compute the Transmission Coefficient:**\
    Finally, calculate the transmission probability as:\
    $$
    T = |t|^2 \cdot \frac{k_N}{k_0}.
    $$


In [ ]:
# install required libraries (run once)
import sys
print("Python executable:", sys.executable)

!{sys.executable} -m pip install -q --upgrade numpy scipy matplotlib ipython ipywidgets widgetsnbextension jupyterlab_widgets plotly nbformat tqdm joblib

import ipywidgets as widgets
print("ipywidgets:", widgets.__version__)

import nbformat
print("nbformat:", nbformat.__version__)

In [ ]:
import numpy as np
import numpy.linalg as la
import math
from scipy.signal import find_peaks
from joblib import Parallel, delayed
from scipy.constants import physical_constants
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from ipywidgets import interact_manual

###############################################################################
# Global Constants and Unit Conversions
###############################################################################
# 1 Bohr = 0.52917721067 Angstrom, so 1 Angstrom = 1/0.52917721067 Bohr ~ 1.88973
bohr_radius     = physical_constants['Bohr radius'][0]  # in miter
angstrom_radius = 1e-10                                 # in miter


hartree_energy  = physical_constants['Hartree energy'][0]   # in Jule
ev_energy       = physical_constants['electron volt'][0]    # in Jule


hbar = physical_constants["reduced Planck constant"][0]     # 1.0545718176461565e-34 J s
h    = hbar * 2 * np.pi

q    = physical_constants["elementary charge"][0]           # 1.602176634e-19 C
kB_eV   = physical_constants["Boltzmann constant in eV/K"][0]  # 8.617333262145179e-05 eV K^-1



###############################################################################
# 1. Basic Helpers & Potential Construction
###############################################################################

def potential(  num_barriers: int,
                barrier_height: float,
                len_barrier: float,
                len_well: float,
                dx: float,
                bias_eV: float,
                pot_shape: int):
    """
    Create a 1D potential array for a finite chain of 'num_barriers' barriers,
    each barrier of given width and height, separated by wells (V=0).
    The overall domain spans from 0 to (len_left_and_well + num_barriers*(len_barrier+len_well) + len_right),
    where the leftmost and rightmost margins have 0 potential.
    
    A triangular bias is applied only in the region containing the barrier structure 
    (from x = len_left_and_well to x = len_left_and_well + num_barriers*(len_barrier+len_well)).
    The bias is 0 at the boundaries and reaches bias_eV at the midpoint.
    """
    len_left_and_well = 2 + len_well  # 2 for electrode, len_well for symmetry of potential
    len_right         = 0 + 2
    len_cell          = len_barrier + len_well
    len_cell_total    = num_barriers * len_cell
    len_total         = len_left_and_well + len_cell_total + len_right

    Nx  = int(np.ceil(len_total / dx))
    x   = np.linspace(0, len_total, Nx)
    V = np.zeros(Nx)
    
    # Build the periodic potential in the barrier region
    for b in range(num_barriers):

        start_x = len_left_and_well + b * len_cell
        end_x   = len_left_and_well + b * len_cell + len_barrier

        # Square well
        if pot_shape == 0: 
            start_idx = int(start_x / dx)
            end_idx   = int(end_x / dx)
            V[start_idx:end_idx] = barrier_height
            
        # Triangualr well
        elif pot_shape == 1:
            start_idx = int(start_x / dx)
            end_idx   = int(end_x / dx)
            for i in range(start_idx, end_idx +1):
                V[i] = barrier_height * (i - start_idx) / (end_idx - start_idx)

        # Parabolic (concave) (U-shaped well)
        elif pot_shape == 2:
            start_idx = int(start_x / dx)
            end_idx   = int(end_x / dx)
            for i in range(start_idx, end_idx +1):
                V[i] = barrier_height * ( ( i - (start_idx + end_idx)/2 ) / ((end_idx - start_idx)/2 ) )**2


        # Parabolic (convex) (Approximation of Coulomic well, Y-shaped well, 1-r**2)
        elif pot_shape == 3:
            start_idx = int(start_x / dx)
            end_idx   = int(end_x / dx)
            for i in range(start_idx, end_idx +1):
                V[i] = barrier_height * (1 - ( ( i - (start_idx + end_idx)/2 ) / ((end_idx - start_idx)/2 ) )**2)

        # Custom potential   Design your potential :)
        elif pot_shape == 4:
            start_idx = int(start_x / dx)
            end_idx   = int(end_x / dx)
            for i in range(start_idx, end_idx +1):
                ### 

                V[i] = 0   # Edit here
                
                ###


    # Apply a linear bias over the barrier region
    if abs(bias_eV) > 1e-12:
        bias = np.zeros(Nx)
        x_left = len_left_and_well - (len_well)
        x_right = len_left_and_well + len_cell_total
        for i, xi in enumerate(x):
            if xi < x_left:
                bias[i] = 0
            elif xi > x_right:
                bias[i] = bias_eV
            else:
                bias[i] = bias_eV * ((xi - x_left) / (x_right - x_left))
        V = V + bias
    return x, V

def k_local(E, V):
    """
    Local wave number for E - V in 1D (atomic units: ħ=1, m=1).
      k = sqrt(2*(E - V)), real if E>V, imaginary if E<V.
    """
    delta = E - V
    if delta >= 0:
        return np.sqrt(2*delta + 0j)
    else:
        return 1j * np.sqrt(-2*delta)


def full_transfer_matrix(x, V, E, dx):
    """
    Compute the full transfer matrix for an open-boundary potential V(x)
    by stepping through the grid cell-by-cell.
    """
    
    Nx = len(x)
    M_total = np.eye(2, dtype=complex)
    
    k_left  = k_local(E, 0.0)
    k_right = k_local(E, 0.0)
    k_layer0 = k_local(E, V[0])
    
    M_int_left = 0.5 * np.array([[1 + k_layer0 / k_left, 1 - k_layer0 / k_left],
                                [1 - k_layer0 / k_left, 1 + k_layer0 / k_left]], dtype=complex)
    M_total = M_int_left @ M_total
    
    for i in range(Nx - 1):
        k_i     = k_local(E, V[i])
        k_ip1   = k_local(E, V[i+1])
        M_prop  = np.array([[np.exp(1j*k_i*dx),  0],
                           [0, np.exp(-1j*k_i*dx)]], dtype=complex)
        M_total = M_prop @ M_total
        M_int = 0.5 * np.array([[1 + k_ip1 / k_i, 1 - k_ip1 / k_i],
                                [1 - k_ip1 / k_i, 1 + k_ip1 / k_i]], dtype=complex)
        M_total = M_int @ M_total
    
    k_last = k_local(E, V[-1])
    M_prop_last = np.array([[np.exp(1j*k_last*dx),  0],
                            [0, np.exp(-1j*k_last*dx)]], dtype=complex)
    M_total = M_prop_last @ M_total
    M_int_right = 0.5 * np.array([[1 + k_right / k_last, 1 - k_right / k_last],
                                  [1 - k_right / k_last, 1 + k_right / k_last]], dtype=complex)
    M_total = M_int_right @ M_total
    
    return M_total

def transmission_coefficient_open(M):
    """
    Compute transmission coefficient T = 1 / |M[0,0]|^2.
    """
    return 1.0 / np.abs(M[0,0])**2

def fermi_dirac(E_array, mu_f, temp):
    x = (E_array - mu_f) / (eff_kB * temp)
    return 1.0 / (1.0 + np.exp(x))

def calc_current(T_tmm, emin, emax, e_resolution, bias, mu_f, temp):
    '''
    Landauer-Büttiker formula

    I=2e^2/h ∫T(E)[fL-fR]dE (E is eV)

    It only calculates current from left to right

    ref - Phys. Rev. B 41, 7906(R) (1990)
    '''
    E_array = np.linspace(emin, emax, e_resolution)
    idx_l = np.where(E_array >= 0)[0][0]
    idx_r = np.where(E_array >= 0 + bias)[0][0]

    fl = fermi_dirac(E_array, mu_f, temp)
    fr = fermi_dirac(E_array, mu_f + bias, temp)

    window = fl - fr
    # Conduction band minimum
    if bias < 0 :
        window[0:idx_l] = 0 
    else:
        window[0:idx_r] = 0 
    
    degeneracy = 1  # (spin, ...)
    coeff = (degeneracy * q**2) / h   # A/eV
    I = coeff * np.trapezoid(T_tmm * window, E_array * eff_har2ev)  # [A]

    return I

###############################################################################
# 2. Compute Transmission and Find Resonance Peaks (Parallelized)
###############################################################################
def compute_transmission(num_barriers:   int,
                         barrier_height: float,
                         barrier_width:  float,
                         well_width: float,
                         dx:    float,
                         emin:  float, 
                         emax:  float, 
                         e_resolution: int,
                         bias:   float,
                         pot_shape: int):
    """
    For a given number of barriers, build the open-boundary potential,
    compute T(E) via TMM (in parallel), and identify resonance peaks.
    
    Returns a dict with:
      "x": coordinate array,
      "V": potential array,
      "T_tmm": transmission array (clamped to [0,1]),
      "peaks_tmm": resonance peak energies.
    """
    E_array = np.linspace(emin, emax, e_resolution)

    x, V = potential(num_barriers, barrier_height, barrier_width, well_width, dx,
                                  bias, pot_shape)
    
    def compute_T(E):
        M = full_transfer_matrix(x, V, E, dx)
        T_val = transmission_coefficient_open(M)
        return max(min(T_val, 1.0), 1e-12)
    
    
    T_tmm = Parallel(n_jobs=-1)(delayed(compute_T)(E) for E in E_array)
    T_tmm = np.array(T_tmm)
    T_tmm[0] = 0
   
    logT = np.log1p(T_tmm)
    pkidx, _ = find_peaks(logT, prominence=1e-3)    # Tolarance
    Epeaks_tmm = E_array[pkidx]
  
    return dict(x=x, V=V, T_tmm=T_tmm, peaks_tmm=Epeaks_tmm)

###############################################################################
# 3. Plot Using Plotly: Potential, Transmission, and Resonance Peaks vs. Occurrence
###############################################################################
def plot_transsmission(result, num_barriers, emin, emax, e_resolution):
    """
    Plot three panels:
      (1) Potential with horizontal lines at each resonance energy.
      (2) Transmission (log scale on x-axis) vs. Energy with markers at resonances.
      (3) Scatter plot of Resonance Peak Energy vs. Occurrence (integer index).
    """
    E_array = np.linspace(emin, emax, e_resolution)
    
    x = (result["x"] - 0.5 * np.max(result["x"])) / eff_ang2bohr    # center at 0
    V = result["V"]
    T_tmm = result["T_tmm"]
    pk_tmm = result["peaks_tmm"]  # resonance energies
    occurrence = np.arange(1, len(pk_tmm)+1)
    
    # Generate colors for each peak using Plotly's colorscale
    if len(pk_tmm) > 1:
        colors = px.colors.sample_colorscale("Rainbow", [i/(len(pk_tmm)-1) for i in range(len(pk_tmm))])
    elif len(pk_tmm) == 1:
        colors = [px.colors.sample_colorscale("Rainbow", [0.5])[0]]
    else:
        colors = []
    
    ### Create subplots with 1 row and 3 columns
    fig = make_subplots(rows=1, cols=3,)
    
    ### Left panel: Potential curve and horizontal lines at resonance energies
    fig.add_trace(go.Scatter(x=x, y=V * eff_har2ev,
                            mode='lines', line=dict(color="black"), name="Potential", showlegend=False),
                row=1, col=1)
    
    # Add horizontal lines for each resonance energy
    for i, epeak in enumerate(pk_tmm):
        fig.add_shape(type="line",
                      x0=x[0], x1=x[-1],
                      y0=epeak * eff_har2ev, y1=epeak * eff_har2ev,
                      line=dict(color=colors[i], dash="dash"), row=1, col=1)
    
    fig.update_yaxes(range=[emin * eff_har2ev, emax * eff_har2ev], row=1, col=1)

    ### Center panel: Transmission vs. Energy (x-axis: T, y-axis: Energy), x-scale log
    fig.add_trace(go.Scatter(x=T_tmm, y=E_array * eff_har2ev,
                             mode='lines', line=dict(color="black"), name="TMM", showlegend=False),
                  row=1, col=2)
    
    # Add markers at resonance peaks
    for i, epeak in enumerate(pk_tmm):
        idx = np.argmin(np.abs(E_array - epeak))
        fig.add_trace(go.Scatter(x=[T_tmm[idx]], y=[epeak * eff_har2ev],
                                 mode='markers', marker=dict(color=colors[i], size=8),
                                 name=f"Resonance {i+1}", showlegend=False),
                      row=1, col=2)

    fig.update_layout(title_text=f"Resonance vs. Occurrence for {num_barriers} barriers", 
                      width=1200, height=500, showlegend=False)

    # Add markers at resonance peaks
    for i, epeak in enumerate(pk_tmm):
        idx = np.argmin(np.abs(E_array - epeak))
        fig.add_trace(go.Scatter(x=[T_tmm[idx]], y=[epeak * eff_har2ev],
                                 mode='markers', marker=dict(color=colors[i], size=8),
                                 name=f"Resonance {i+1}"),
                      row=1, col=2)

    fig.update_xaxes(title_text="Transmission", type="linear", range=[0, 1], row=1, col=2)
    fig.update_yaxes(title_text="Energy (eV)", range=[emin * eff_har2ev, emax * eff_har2ev], row=1, col=2)

    # scale secelector for Transmission (log, linear)
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons", direction="right",
                buttons=list([
                    dict(args=[{"xaxis2.type": "linear", "xaxis2.range": [0, 1]}],
                        label="Linear", method="relayout"
                    ),
                    dict(args=[{"xaxis2.type": "log", "xaxis2.range": [np.log10(1e-6), 0]}],
                        label="Log", method="relayout"
                    )
                ]),
                pad={"r": 10, "t": 10}, showactive=True, x=0.5, xanchor="center", y=1.2, yanchor="top"
                )
            ]
    )
    
    ### Right panel: Resonance Peak Energy vs. Occurrence
    fig.add_trace(go.Scatter(x=occurrence, y=pk_tmm * eff_har2ev,
                             mode='markers', marker=dict(color=colors, size=8),
                             name="Resonance Peaks"),
                  row=1, col=3)
    fig.update_xaxes(title_text="Occurrence", range=[0, len(pk_tmm)+1], row=1, col=3)
    fig.update_yaxes(title_text="Resonance Energy (eV)", range=[emin * eff_har2ev, emax * eff_har2ev], row=1, col=3)
    
    fig.update_layout(title_text=f"Resonance vs. Occurrence for {num_barriers} barriers", width=1200, height=500)
    fig.show()

###############################################################################
# 4. Interactive Widget
###############################################################################

slider_layout = widgets.Layout(width='450px')  # Set a fixed width for all sliders
label_style = {'description_width': '200px'}  # Uniform label width for alignment

@interact_manual(
    num_barriers        = widgets.IntSlider(min=1, max=40, step=1, value=1, 
                                    description='# of Barriers : ',
                                    style=label_style, layout=slider_layout),
    barrier_height_eV   = widgets.FloatSlider(min=0.5, max=20, step=0.1, value=3, 
                                    description='Barrier Height [eV] : ',
                                    style=label_style, layout=slider_layout),
    barrier_width_ang   = widgets.FloatSlider(min=0.1, max=500.0, step=0.1, value=3.0,
                                    description='Barrier Width [Ang] : ',
                                    style=label_style, layout=slider_layout),
    well_width_ang      = widgets.FloatSlider(min=0.1, max=150.0, step=0.1, value=3.0,
                                    description='Well Width [Ang] : ',
                                    style=label_style,  layout=slider_layout),
    bias_eV             = widgets.FloatSlider(min=-4, max=4, step=0.1, value=0.0,
                                    description='Linear Bias [eV] : ',
                                    style=label_style,  layout=slider_layout),
    mu_f_eV             = widgets.FloatSlider(min=0, max=4, step=0.1, value=0.0,
                                    description='Fermi level of electrode [eV] : ',
                                    style=label_style,  layout=slider_layout),
    pot_shape           = widgets.Dropdown(options=[
                                        ('Square Well', 0),
                                        ('Triangular Well', 1),
                                        ('Parabolic Well (concave)', 2),
                                        ('Parabolic well (convex)', 3),
                                        ('Custom potential', 4),],
                                    value=0, description='Potential Shape: ',
                                    style=label_style, layout=slider_layout),
    m_eff               = widgets.Dropdown(options=[
                                        ('GaAs : 0.07',  0.07),
                                        ('Silicon : 1.08', 1.08),
                                        ('Germanium : 0.55', 0.55),
                                        ('Bare mass : 1.00',  1.00),],
                                    value=1.00, description='Effective mass', 
                                    readout=True, style=label_style, layout=slider_layout),
)

def main(num_barriers, barrier_height_eV, barrier_width_ang, well_width_ang, bias_eV, mu_f_eV, pot_shape, m_eff):
    # Variables
    dx = 0.05   # Bohr
    e_resolution = 1500
    temp = 300  # Kelbin

    ### Effective mass atomic unit conversion
    global eff_bohr
    global eff_ang2bohr
    global eff_har
    global eff_har2ev
    global eff_kB
    
    eff_bohr     = bohr_radius / m_eff
    eff_ang2bohr = 1e-10 / eff_bohr
    eff_har      = hartree_energy * m_eff
    eff_har2ev   = eff_har / ev_energy

    eff_kB       = kB_eV / eff_har2ev   # effective Hartree / Kelbin
    
    bias            = bias_eV           / eff_har2ev
    mu_f            = mu_f_eV           / eff_har2ev
    barrier_height  = barrier_height_eV / eff_har2ev

    barrier_width   = barrier_width_ang * eff_ang2bohr
    well_width      = well_width_ang    * eff_ang2bohr

    

    if bias > 0:
        emin = 0
        emax = (barrier_height + bias) * 3
    else:
        emin = 0.0 + bias
        emax = (barrier_height) * 3

    # Compute transmission using TMM (parallelized)
    print("\n"*2+"(1) Calculation")
    result = compute_transmission(num_barriers, barrier_height, barrier_width, well_width, dx, emin, emax, e_resolution, bias, pot_shape)
    
    # Compute current
    I = calc_current(result["T_tmm"], emin, emax, e_resolution, bias, mu_f, temp)
    print(f'\nCurrent is {I:.6e} A at voltage {bias_eV} eV')
    print("\n"*2+"(1) >> done!!!")

    # Plot
    print("\n"*2+"(2) Visualization)")
    plot_transsmission(result, num_barriers, emin, emax, e_resolution)
    print("\n"*2+"(2) >> done!!!")

if __name__ == "__main__":
    readme = r"""
             
             V
             ↑
             │    Left     distance width           Right
             │             <------> <---->
      height │             ┌-------┐     ┌-------┐            
             │             │       │     │       │ 
 Fermi level │    -----    │       │     │       │  -----
             │             │       │     │       │ 
           0 │ ------------┘       └-----┘       └-----------
             │------------------------┼----------------------> x [Ang]
                                   center
        
    """
    print(readme)
    pass
